In [2]:
from pathlib import Path
import delta as delta_lib
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from delta import *
from delta.tables import *

warehouse_dir = Path("/home/jovyan/work/spark-warehouse").resolve()
warehouse_dir.mkdir(parents=True, exist_ok=True)
caminho_base = f"file://{warehouse_dir.as_posix()}"
caminho_notas = f"{caminho_base}/notas_delta"

# ==========================================
# 1. LIGANDO O MOTOR
# ==========================================
print("Iniciando o motor DELTA...")
builder = SparkSession.builder.appName("DeltaLake_Lab") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.warehouse.dir", caminho_base) \
    .config("spark.databricks.delta.retentionDurationCheck.enabled", "false")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark version: {spark.version}")
print(f"Delta version: {getattr(delta_lib, '__version__', 'unknown')}")
print("✅ Motor DELTA ligado!\n")

# ==========================================
# 2. LABORATÓRIO: INSERT, UPDATE E DELETE
# ==========================================
print("--- 1. INSERT (Criando as tabelas) ---")
dados_notas = [
    (101, 1, "Arquitetura de Dados", 8.5),
    (102, 2, "Arquitetura de Dados", 6.0)
]
schema_notas = StructType([
    StructField("id_nota", IntegerType(), True), StructField("id_aluno", IntegerType(), True),
    StructField("disciplina", StringType(), True), StructField("valor_nota", FloatType(), True)
])

df_notas = spark.createDataFrame(dados_notas, schema_notas)

df_notas.write.format("delta").mode("overwrite").save(caminho_notas)
df_notas.show()

print("--- 2. UPDATE (Mudando a nota do aluno 1 para 9.5) ---")
tabela_notas_delta = DeltaTable.forPath(spark, caminho_notas)
tabela_notas_delta.update(condition = "id_aluno = 1", set = { "valor_nota": "9.5" })
tabela_notas_delta.toDF().show()

print("--- 3. DELETE (Removendo o aluno 2) ---")
tabela_notas_delta.delete(condition = "id_aluno = 2")
tabela_notas_delta.toDF().show()

Iniciando o motor DELTA...
Spark version: 3.5.0
Delta version: unknown
✅ Motor DELTA ligado!

--- 1. INSERT (Criando as tabelas) ---
+-------+--------+--------------------+----------+
|id_nota|id_aluno|          disciplina|valor_nota|
+-------+--------+--------------------+----------+
|    101|       1|Arquitetura de Dados|       8.5|
|    102|       2|Arquitetura de Dados|       6.0|
+-------+--------+--------------------+----------+

--- 2. UPDATE (Mudando a nota do aluno 1 para 9.5) ---
+-------+--------+--------------------+----------+
|id_nota|id_aluno|          disciplina|valor_nota|
+-------+--------+--------------------+----------+
|    101|       1|Arquitetura de Dados|       9.5|
|    102|       2|Arquitetura de Dados|       6.0|
+-------+--------+--------------------+----------+

--- 3. DELETE (Removendo o aluno 2) ---
+-------+--------+--------------------+----------+
|id_nota|id_aluno|          disciplina|valor_nota|
+-------+--------+--------------------+----------+
|    